In [12]:
from datasets import load_dataset
from tqdm.auto import tqdm
import time
from utils import QueryModelError, QueryJudgeError, GetJudgePrompt, SaveJsonl
from run import run
from joblib import Parallel, delayed
import numpy as np
import random
from utils import LoadJson
import os
import matplotlib.pyplot as plt
import re
from typing import List

def list_files(directory):
    return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]


def extract_numbers(text: str) -> List[float]:
    """
    Return a list of numbers found in *text*.
    Handles integers, decimals, optional +/- signs, and scientific notation.

    >>> extract_numbers("The price went from -3.2% to 4.50% in 2e1 days.")
    [-3.2, 4.5, 20.0]
    """
    # Regex matches:
    #   optional sign (+/-)
    #   digits with optional decimal part
    #   optional exponent (e.g., 2e1, 3E-4)
    pattern = r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?'

    return [float(match) for match in re.findall(pattern, text)]

In [13]:
models =  ['o1-preview-2024-09-12',
           'gpt-4o-mini-2024-07-18',
           'gpt-4o-2024-05-13',
           'claude-3-5-sonnet-20241022',
           'gemini-1.5-pro-002',
           'gemini-1.5-flash-002',
           'gemini-1.5-flash-8b-001',
           "mistral-large-2407",
           "mistral-small-2409",
           "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
           "meta-llama/Meta-Llama-3.1-405B-Instruct-Turbo",
           "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
           'google/gemma-2-27b-it',
           'Qwen/Qwen2.5-72B-Instruct',
           'Qwen/Qwen2.5-32B-Instruct',
           'Qwen/Qwen2.5-7B-Instruct',
           'google/gemma-2-9b-it',
           'google/gemma-2-2b-it',
           "ibm-granite/granite-3.0-8b-instruct"]


benchmarks = ['math','musr','gpqa','mmlu-pro','bbh','ifeval'] #'

In [14]:
runs = []
for m in models:
    for b in benchmarks:
        runs.append([m,b])

In [ ]:
n_jobs   = -1       
backend  = "loky"     
verbose  = 10          

results = Parallel(n_jobs=n_jobs, backend=backend, verbose=verbose)(
    delayed(run)(r[1], r[0])                 # what to run
    for r in runs                    # iterable of inputs
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
100%|██████████| 199/199 [00:00<00:00, 5616.87it/s]it/s].05it/s]
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    4.6s
100%|██████████| 199/199 [00:00<00:00, 3702.13it/s]5,  1.52it/s]
[Parallel(n_jobs=-1)]: Done  13 tasks      | elapsed:    5.3s
100%|██████████| 199/199 [00:00<00:00, 4155.28it/s]it/s]
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    6.0s
100%|██████████| 199/199 [00:00<00:00, 7006.23it/s]
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:    6.9s
100%|██████████| 199/199 [00:00<00:00, 6489.95it/s]9,  1.76it/s]
[Parallel(n_jobs=-1)]: Done  44 tasks      | elapsed:    8.1s
100%|██████████| 199/199 [00:00<00:00, 4337.21it/s]
[Parallel(n_jobs=-1)]: Done  57 tasks      | elapsed:   10.1s
100%|██████████| 199/199 [00:00<00:00, 4803.86it/s]
[Parallel(n_jobs=-1)]: Done  70 tasks      | elapsed:   15.9s
100%|██████████| 199/199 [00:00<00:00, 5366.63it/s]9,  2.54it/s]
[Parallel(n_jobs

## Checking missing

In [8]:
missing = {}
for m in models:
    missing[m] = {}
    for b in benchmarks:
        missing[m][b] = {'resp':0, 'score':0}
        path = f'results/{b.replace("/",".").replace("_",".")}/{m.replace("/",".").replace("_",".")}/'
        file_names = list_files(path)
        for i in range(len(file_names)):
            file_name = path+file_names[i]
            d = LoadJson(file_name)
            response_na = d['model_response'] == "NA"
            scores_na = d['scores'] == "NA"

            missing[m][b]['resp'] += 1*response_na
            missing[m][b]['score'] += 1*scores_na

missing

{'o1-preview-2024-09-12': {'math': {'resp': 0, 'score': 0},
  'musr': {'resp': 0, 'score': 0},
  'gpqa': {'resp': 0, 'score': 0},
  'mmlu-pro': {'resp': 0, 'score': 0},
  'bbh': {'resp': 0, 'score': 0},
  'ifeval': {'resp': 0, 'score': 0}},
 'gpt-4o-mini-2024-07-18': {'math': {'resp': 0, 'score': 0},
  'musr': {'resp': 0, 'score': 0},
  'gpqa': {'resp': 0, 'score': 0},
  'mmlu-pro': {'resp': 0, 'score': 0},
  'bbh': {'resp': 0, 'score': 0},
  'ifeval': {'resp': 0, 'score': 0}},
 'gpt-4o-2024-05-13': {'math': {'resp': 0, 'score': 0},
  'musr': {'resp': 0, 'score': 0},
  'gpqa': {'resp': 0, 'score': 0},
  'mmlu-pro': {'resp': 0, 'score': 0},
  'bbh': {'resp': 0, 'score': 0},
  'ifeval': {'resp': 0, 'score': 0}},
 'claude-3-5-sonnet-20241022': {'math': {'resp': 0, 'score': 0},
  'musr': {'resp': 0, 'score': 0},
  'gpqa': {'resp': 0, 'score': 0},
  'mmlu-pro': {'resp': 0, 'score': 0},
  'bbh': {'resp': 0, 'score': 0},
  'ifeval': {'resp': 0, 'score': 0}},
 'gemini-1.5-pro-002': {'math': {'

## Process data

In [9]:
m = 'google.gemma-2-2b-it'

ids = {}
Ys = {}
for b in benchmarks:
    path = f'results/{b.replace("/",".").replace("_",".")}/{m.replace("/",".").replace("_",".")}/'
    file_names = list_files(path)
    ids[b] = np.unique([int(f.replace(".json","").split("_")[-1]) for f in file_names]).tolist()
    Ys[b] = -99.0*np.ones((len(models), len(ids[b])))

subjects = {}
for b in benchmarks:
    subjects[b] = []
    for l,m in enumerate(models):
        path = f'results/{b.replace("/",".").replace("_",".")}/{m.replace("/",".").replace("_",".")}/'
        for i,id in enumerate(ids[b]):
            file_name = path+b+"_"+m.replace("/",".").replace("_",".")+"_"+str(id)+".json"
            try:
                d = LoadJson(file_name)
                if l==0:
                    subjects[b].append(d['subject'])
                if b=='ifeval':
                    Ys[b][l,i] = np.mean(d['scores']['inst_level_loose_acc'])==1
                else:
                    idx = d['scores'].find("correctness_score")
                    if idx>=0:
                        Ys[b][l,i] = float(extract_numbers(d['scores'][idx:])[0])
            except:
                pass
    print(b,(Ys[b]==-99).sum())

math 25
musr 0
gpqa 11
mmlu-pro 11
bbh 0
ifeval 2


In [10]:
for b in benchmarks:
    print(b)
    print(np.unique(subjects[b], return_counts=True))

math
(array(['Algebra_Level-5', 'Counting & Probability_Level-5',
       'Geometry_Level-5', 'Intermediate Algebra_Level-5',
       'Number Theory_Level-5', 'Prealgebra_Level-5',
       'Precalculus_Level-5'], dtype='<U30'), array([93, 37, 40, 84, 46, 58, 41]))
musr
(array(['murder_mysteries', 'object_placements', 'team_allocation'],
      dtype='<U17'), array([132, 135, 132]))
gpqa
(array(['main'], dtype='<U4'), array([399]))
mmlu-pro
(array(['biology', 'business', 'chemistry', 'computer science',
       'economics', 'engineering', 'health', 'history', 'law', 'math',
       'other', 'philosophy', 'physics', 'psychology'], dtype='<U16'), array([24, 28, 38, 14, 28, 33, 28, 12, 36, 44, 30, 16, 42, 26]))
bbh
(array(['boolean_expressions', 'causal_judgement', 'date_understanding',
       'disambiguation_qa', 'formal_fallacies', 'geometric_shapes',
       'hyperbaton', 'logical_deduction_five_objects',
       'logical_deduction_seven_objects',
       'logical_deduction_three_objects', 'movi

In [11]:
Ys['models']=models
np.save("results/Ys.npy",Ys)